# BÁO CÁO THỰC THI MÃ NGUỒN MACHINE LEARNING — CHU TRÌNH CRISP-DM
## ĐỀ TÀI: HỆ THỐNG ĐỊNH LƯỢNG MACHINE LEARNING & QUÉT VI MÔ DÒNG TIỀN THỂ CHẾ PHÁI SINH CỔ PHIẾU MỸ (TRADFI US STOCK USD-M 24/7)

- **Cơ sở đào tạo:** Trường Đại học Công nghệ Thông tin — ĐHQG TP.HCM (UIT)
- **Khoa:** Khoa Khoa học và Kỹ thuật Thông tin
- **Đơn vị quản lý / Hệ đào tạo:** Văn bằng 2 — Trung tâm Phát triển Công nghệ Thông tin (CITD)
- **Học viên thực hiện:** NGUYỄN DƯƠNG MẠNH SINH — MSSV: 24210169
- **Lớp khóa học:** Lớp SIC 2026: AI1 UFM
- **Giảng viên hướng dẫn:** ThS. Trần Mạnh Tường
- **Thời gian thực hiện:** Tháng 06/2026 – Tháng 10/2026
- **Trạng thái đóng góp:** 100% — Đã hoàn thành
- **Tập dữ liệu thực nghiệm:** data/NVDAUSDT_5m_288candles_20260827.csv (Nến vi mô 5M từ Binance Futures)
- **Kho lưu trữ GitHub (Public):** [https://github.com/sinhnguyen1411/us-stock-machine-learning-scanners](https://github.com/sinhnguyen1411/us-stock-machine-learning-scanners)


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

print('Toàn bộ thư viện cốt lõi đã sẵn sàng!')


Toàn bộ thư viện cốt lõi đã sẵn sàng!


In [2]:
# GIAI ĐOẠN 1 & 2: DATA UNDERSTANDING
df = pd.read_csv('data/NVDAUSDT_5m_288candles_20260827.csv')
print(f'Kích thước tập dữ liệu nến 5M: {df.shape[0]} dòng x {df.shape[1]} cột')
print('Các cột thuộc tính chính:', list(df.columns[:10]))
print('\nKiểm tra dữ liệu khuyết thiếu (Missing Values):')
print(df[['open', 'high', 'low', 'close', 'volume', 'quote_volume', 'buy_ratio']].isnull().sum())
df[['open', 'high', 'low', 'close', 'volume', 'buy_ratio', 'is_pump_target']].head()


Kích thước tập dữ liệu nến 5M: 288 dòng x 21 cột
Các cột thuộc tính chính: ['timestamp', 'open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'quote_volume', 'trades']

Kiểm tra dữ liệu khuyết thiếu (Missing Values):
open            0
high            0
low             0
close           0
volume          0
quote_volume    0
buy_ratio       0
dtype: int64


In [3]:
# GIAI ĐOẠN 3: DATA PREPARATION & FEATURE ENGINEERING (5 ĐẶC TRƯNG VI MÔ)
# 1. Đường giá thể chế VWAP
cum_vol = df['volume'].cumsum()
cum_vol_price = (df['volume'] * (df['high'] + df['low'] + df['close']) / 3.0).cumsum()
df['vwap'] = cum_vol_price / np.maximum(cum_vol, 1e-4)

# Đặc trưng 1: VWAP Distance (%)
df['f1_vwap_dist'] = ((df['close'] - df['vwap']) / df['vwap']) * 100.0

# Đặc trưng 2: Volume Spike vs 20-EWMA
df['ewma_vol_20'] = df['volume'].ewm(span=20, adjust=False).mean()
df['f2_vol_spike'] = df['volume'] / np.maximum(df['ewma_vol_20'], 1e-4)

# Đặc trưng 3: Wyckoff Range Coiling 24 nến (2 giờ gần nhất)
rolling_high = df['high'].rolling(window=24, min_periods=1).max()
rolling_low = df['low'].rolling(window=24, min_periods=1).min()
df['f3_wyckoff_coiling'] = ((rolling_high - rolling_low) / np.maximum(rolling_low, 1e-4)) * 100.0

# Đặc trưng 4: Taker Buy Volume Ratio
df['f4_taker_buy_ratio'] = df['buy_ratio'].fillna(0.5)

# Đặc trưng 5: ATR Relative Volatility
tr = np.maximum(df['high'] - df['low'], np.maximum(abs(df['high'] - df['close'].shift(1)), abs(df['low'] - df['close'].shift(1))))
df['atr_14'] = tr.rolling(window=14, min_periods=1).mean()
df['f5_atr_rel'] = (df['atr_14'] / df['close']) * 100.0

features = ['f1_vwap_dist', 'f2_vol_spike', 'f3_wyckoff_coiling', 'f4_taker_buy_ratio', 'f5_atr_rel']
print('5 đặc trưng vi mô đã được vector hóa thành công!')
df[features].describe().T[['mean', 'std', 'min', '50%', 'max']]


5 đặc trưng vi mô đã được vector hóa thành công!


In [4]:
# GIAI ĐOẠN 3: PHÂN TÍCH KHÁM PHÁ DỮ LIỆU (EDA)
skew_vol = df['volume'].skew()
kurt_vol = df['volume'].kurt()
print(f'Hệ số lệch khối lượng (Skewness): {skew_vol:.2f}')
print(f'Hệ số nhọn khối lượng (Kurtosis): {kurt_vol:.2f}')

corr = df[features + ['is_pump_target']].corr()
print('\nMa trận tương quan với biến mục tiêu (is_pump_target):')
print(corr['is_pump_target'].sort_values(ascending=False))


Hệ số lệch khối lượng (Skewness): 10.15
Hệ số nhọn khối lượng (Kurtosis): 124.91

Ma trận tương quan với biến mục tiêu (is_pump_target):
is_pump_target        1.000000
f5_atr_rel            0.327792
f3_wyckoff_coiling    0.223613
f2_vol_spike          0.146064
f4_taker_buy_ratio    0.018601
f1_vwap_dist         -0.135393
Name: is_pump_target, dtype: float64


In [5]:
# GIAI ĐOẠN 4: PHÂN CHIA CHUỖI THỜI GIAN CHỐNG RÒ RỈ (PURGED SPLIT)
split_idx = int(len(df) * 0.70)
X = df[features]
y = df['is_pump_target']

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f'Tập Huấn luyện (Train): {len(X_train)} nến | Tỷ lệ nhãn dương: {y_train.mean():.1%}')
print(f'Tập Kiểm định (Test):  {len(X_test)} nến | Tỷ lệ nhãn dương: {y_test.mean():.1%}')


Tập Huấn luyện (Train): 201 nến | Tỷ lệ nhãn dương: 3.5%
Tập Kiểm định (Test):  87 nến | Tỷ lệ nhãn dương: 2.3%


In [6]:
# GIAI ĐOẠN 4: MÔ HÌNH CƠ SỞ (BASELINE HEURISTIC)
# Quy tắc: Volume Spike > 2.0x và Giá > VWAP
y_pred_base = ((X_test['f2_vol_spike'] > 2.0) & (X_test['f1_vwap_dist'] > 0.5)).astype(int)
acc_base = accuracy_score(y_test, y_pred_base)
prec_base = precision_score(y_test, y_pred_base, zero_division=0)
rec_base = recall_score(y_test, y_pred_base, zero_division=0)
f1_base = f1_score(y_test, y_pred_base, zero_division=0)

print('--- HIỆU NĂNG MÔ HÌNH CƠ SỞ (BASELINE) ---')
print(f'Accuracy:  {acc_base:.3f}')
print(f'Precision: {prec_base:.3f}')
print(f'Recall:    {rec_base:.3f}')
print(f'F1-Score:  {f1_base:.3f}')


--- HIỆU NĂNG MÔ HÌNH CƠ SỞ (BASELINE) ---
Accuracy:  0.943
Precision: 0.000
Recall:    0.000
F1-Score:  0.000


In [7]:
# GIAI ĐOẠN 4: MÔ HÌNH RANDOM FOREST CLASSIFIER
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

print('--- HIỆU NĂNG MÔ HÌNH RANDOM FOREST ---')
print(f'Accuracy:  {accuracy_score(y_test, y_pred_rf):.3f}')
print(f'Precision: {precision_score(y_test, y_pred_rf, zero_division=0):.3f}')
print(f'Recall:    {recall_score(y_test, y_pred_rf, zero_division=0):.3f}')
print(f'F1-Score:  {f1_score(y_test, y_pred_rf, zero_division=0):.3f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_proba_rf):.3f}')


--- HIỆU NĂNG MÔ HÌNH RANDOM FOREST ---
Accuracy:  0.782
Precision: 0.095
Recall:    1.000
F1-Score:  0.174
ROC-AUC:   0.859


In [8]:
# GIAI ĐOẠN 4: MÔ HÌNH LIGHTGBM CLASSIFIER (OPTIMIZED)
lgb_model = lgb.LGBMClassifier(
    n_estimators=150,
    learning_rate=0.035,
    num_leaves=31,
    max_depth=5,
    subsample=0.82,
    colsample_bytree=0.75,
    random_state=42,
    verbose=-1
)
lgb_model.fit(X_train, y_train)
y_pred_lgb = lgb_model.predict(X_test)
y_proba_lgb = lgb_model.predict_proba(X_test)[:, 1]

print('--- HIỆU NĂNG MÔ HÌNH LIGHTGBM (TUNED) ---')
print(f'Accuracy:  {accuracy_score(y_test, y_pred_lgb):.3f}')
print(f'Precision: {precision_score(y_test, y_pred_lgb, zero_division=0):.3f}')
print(f'Recall:    {recall_score(y_test, y_pred_lgb, zero_division=0):.3f}')
print(f'F1-Score:  {f1_score(y_test, y_pred_lgb, zero_division=0):.3f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_proba_lgb):.3f}')
print(f'PR-AUC:    {average_precision_score(y_test, y_proba_lgb):.3f}')


--- HIỆU NĂNG MÔ HÌNH LIGHTGBM (TUNED) ---
Accuracy:  0.724
Precision: 0.077
Recall:    1.000
F1-Score:  0.143
ROC-AUC:   0.832
PR-AUC:    0.083


In [9]:
# GIAI ĐOẠN 5: ĐÁNH GIÁ MA TRẬN NHẦM LẪN (CONFUSION MATRIX)
cm = confusion_matrix(y_test, y_pred_lgb)
tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (cm[0,0], 0, 0, 0)

print('--- MA TRẬN NHẦM LẪN CHI TIẾT ---')
print(f'True Negative  (TN - Bỏ qua chuẩn):   {tn}')
print(f'False Positive (FP - Báo động giả):   {fp}')
print(f'False Negative (FN - Bỏ lỡ cơ hội):   {fn}')
print(f'True Positive  (TP - Bắt trúng sóng): {tp}')

# Độ quan trọng của các đặc trưng
importances = pd.Series(lgb_model.feature_importances_, index=features).sort_values(ascending=False)
print('\n--- XẾP HẠNG ĐỘ QUAN TRỌNG ĐẶC TRƯNG (FEATURE IMPORTANCE) ---')
for feat, imp in importances.items():
    print(f'{feat:20s}: {imp:4d} lượt chia nhánh')


--- MA TRẬN NHẦM LẪN CHI TIẾT ---
True Negative  (TN - Bỏ qua chuẩn):   61
False Positive (FP - Báo động giả):   24
False Negative (FN - Bỏ lỡ cơ hội):   0
True Positive  (TP - Bắt trúng sóng): 2

--- XẾP HẠNG ĐỘ QUAN TRỌNG ĐẶC TRƯNG (FEATURE IMPORTANCE) ---
f5_atr_rel          :  296 lượt chia nhánh
f3_wyckoff_coiling  :  247 lượt chia nhánh
f2_vol_spike        :  215 lượt chia nhánh
f1_vwap_dist        :  106 lượt chia nhánh
f4_taker_buy_ratio  :   82 lượt chia nhánh


In [10]:
# GIAI ĐOẠN 6: TRIỂN KHAI THỰC CHIẾN & QUẢN TRỊ RỦI RO
latest_candle = X_test.iloc[-1:]
latest_price = df['close'].iloc[-1]
latest_atr = df['atr_14'].iloc[-1]

# Tính điểm số Machine Learning Composite Score (0 - 100)
raw_prob = lgb_model.predict_proba(latest_candle)[0, 1]
ml_score = round(raw_prob * 100.0, 1)

print('=======================================================')
print('   DEEPALPHA REAL-TIME TRADFI SCANNERS INFERENCE')
print('=======================================================')
print(f'Mã Ticker:            NVDAUSDT')
print(f'Mức giá hiện tại:     ')
print(f'Điểm số ML Composite: {ml_score} / 100.0')

# Kiểm tra cơ chế ngắt mạch Circuit Breaker
simulated_daily_pnl = -12.50 # Giả định mức lỗ trong ngày (-.50)
circuit_breaker_active = simulated_daily_pnl <= -50.0

if circuit_breaker_active:
    print('TRẠNG THÁI: [NGẮT MẠCH CIRCUIT BREAKER KÍCH HOẠT] — Ngừng vào lệnh')
elif ml_score >= 80.0:
    sl = latest_price - (1.5 * latest_atr)
    tp1 = latest_price * 1.025
    tp2 = latest_price * 1.050
    tp3 = latest_price * 1.100
    print('TRẠNG THÁI: [XÁC NHẬN TÍN HIỆU TÍCH LŨY / MỞ VỊ THẾ BUY]')
    print(f' -> Entry Point:        ')
    print(f' -> Dynamic Stop-Loss:   (1.5x ATR buffer: -)')
    print(f' -> Mức 1 (TP1 +2.5%):   [Đóng 40% khối lượng, dời SL hòa vốn]')
    print(f' -> Mức 2 (TP2 +5.0%):   [Đóng 30% khối lượng, bật Trailing Stop]')
    print(f' -> Mức 3 (TP3 +10.0%):  [Đóng 30% khối lượng còn lại]')
else:
    print('TRẠNG THÁI: [THEO DÕI NỀN TÍCH LŨY] — Chưa đủ điều kiện kích hoạt')
print('=======================================================')


   DEEPALPHA REAL-TIME TRADFI SCANNERS INFERENCE
Mã Ticker:            NVDAUSDT
Mức giá hiện tại:     
Điểm số ML Composite: 0.0 / 100.0
TRẠNG THÁI: [THEO DÕI NỀN TÍCH LŨY] — Chưa đủ điều kiện kích hoạt
